# Inheritance in Python: Single, Multiple, MRO, and `super()`

**What you will learn:**
- How **single inheritance** works, with proper use of `super()`
- How **multiple inheritance** works, using a realistic mixin example
- What the **Method Resolution Order (MRO)** is, and how to inspect it
- The classic **diamond problem**, and what goes wrong if you don't use `super()`
- How `super()` fixes it, using Python's cooperative multiple inheritance model


## 1. Single Inheritance

Single inheritance means a class inherits from exactly **one** parent class.

Below, `Dog` inherits from `Animal`. Notice:
- `Dog` reuses `Animal`'s `__init__` via `super().__init__(...)` instead of repeating that code
- `Dog` **overrides** the `speak()` method with its own version
- `Dog` still has access to anything `Animal` defines that it doesn't override (like `describe()`)


In [1]:
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound

    def speak(self):
        return f"{self.name} says {self.sound}"

    def describe(self):
        # Generic method available to ALL animals and their subclasses
        return f"{self.name} is an animal."


class Dog(Animal):
    def __init__(self, name):
        # super().__init__(...) calls Animal's __init__ for us.
        # This avoids repeating "self.name = name" and "self.sound = sound" here.
        super().__init__(name, sound="Woof")

    def speak(self):
        # Overriding speak() to add Dog-specific behaviour,
        # while still reusing Animal's version via super().
        base_message = super().speak()
        return f"{base_message} (a very good dog)"


happy = Dog("Happy")
print(happy.speak())     # uses Dog's overridden speak()
print(happy.describe())  # uses Animal's describe() - Dog didn't need to redefine it


Happy says Woof (a very good dog)
Happy is an animal.


**Key takeaway:** `super()` lets a child class reuse and extend its parent's behaviour,
rather than duplicating code or losing access to it. This is the foundation we'll build on
for multiple inheritance below.


## 2. Multiple Inheritance

Multiple inheritance means a class inherits from **more than one** parent class at once.

A common, genuinely useful pattern is **mixins** — small classes that each add one
specific capability, combined together into a single class.

Below, `FlyingFish` combines `Swimmer` and `Flyer`, gaining both abilities for free.


In [9]:
class Swimmer:
    def swim(self):
        return f"{self.name} is swimming."
    


class Flyer:
    def fly(self):
        return f"{self.name} is flying."
    


class FlyingFish(Swimmer, Flyer):
    def __init__(self, name):
        self.name = name
    


nemo = FlyingFish("Nemo")
print(nemo.swim())   # comes from Swimmer
print(nemo.fly())    # comes from Flyer



# mro - method resolution order

Nemo is swimming.
Nemo is flying.


**Key takeaway:** multiple inheritance lets you compose behaviour from several small,
focused classes rather than building one large class that does everything.

**Is this any use? What's the likely issue if you calss becomes too big but still works?**


## 3. The Method Resolution Order (MRO)

When a class inherits from multiple parents, Python needs a clear, predictable rule
for **which parent's method gets used** if more than one parent defines the same method.

That rule is the **Method Resolution Order (MRO)** — the exact sequence Python searches
through when you call a method, computed using the **C3 linearisation algorithm**:

> 1. Children precede their parents.
> 2. If a class inherits from multiple classes, they are kept in the order specified in the tuple of the base class.

You can always inspect a class's MRO using `ClassName.__mro__` or `ClassName.mro()`.


In [3]:
# The MRO for our FlyingFish example above:
print(FlyingFish.__mro__)

(<class '__main__.FlyingFish'>, <class '__main__.Swimmer'>, <class '__main__.Flyer'>, <class 'object'>)


This reads as: when you call a method on a `FlyingFish` instance, Python looks for it
first on `FlyingFish` itself, then `Swimmer`, then `Flyer`, then `object` (the base of
all Python classes) - stopping at the first match it finds.

### The diamond problem

The MRO becomes really important when you have a **diamond inheritance** shape:
a base class, two classes that both inherit from it, and a final class that inherits
from both of those two. Visually:

```
        Animal
        /    \
     Bird    Mammal
        \    /
        Platypus
```

A platypus is a fun real example here — it has features of both birds and mammals!


In [4]:
class Animal:
    def __init__(self):
        print("Animal.__init__ called")
        self.category = "Animal"

class Bird(Animal):
    def __init__(self):
        print("Bird.__init__ called")
        super().__init__()
        self.lays_eggs = True

class Mammal(Animal):
    def __init__(self):
        print("Mammal.__init__ called")
        super().__init__()
        self.has_fur = True

class Platypus(Bird, Mammal):
    def __init__(self):
        print("Platypus.__init__ called")
        super().__init__()

print("MRO for Platypus:")
for cls in Platypus.__mro__:
    print("  ", cls.__name__)

print()
print("Creating a Platypus instance:")
p = Platypus()


MRO for Platypus:
   Platypus
   Bird
   Mammal
   Animal
   object

Creating a Platypus instance:
Platypus.__init__ called
Bird.__init__ called
Mammal.__init__ called
Animal.__init__ called


Notice something important in the output above: **`Animal.__init__` was only called once**,
even though both `Bird` and `Mammal` inherit from it and both call `super().__init__()`.

This is the entire point of the MRO combined with `super()`: Python uses the MRO to walk
through `Platypus → Bird → Mammal → Animal → object` exactly once each, so every parent's
`__init__` runs, but no parent's `__init__` runs twice. This is called **cooperative
multiple inheritance**.

Let's confirm both `lays_eggs` and `has_fur` were actually set, proving every `__init__`
in the chain genuinely ran:


In [5]:
print("category:  ", p.category)     # set by Animal
print("lays_eggs: ", p.lays_eggs)     # set by Bird
print("has_fur:   ", p.has_fur)       # set by Mammal


category:   Animal
lays_eggs:  True
has_fur:    True


## 4. What Goes Wrong Without `super()`

Now let's break it deliberately. Instead of `super().__init__()`, we'll call each
parent's `__init__` **directly by name** - a common beginner mistake.

This is the exact same diamond shape as before, just with `super()` removed.


In [6]:
class AnimalBad:
    def __init__(self):
        print("AnimalBad.__init__ called")
        self.category = "Animal"

class BirdBad(AnimalBad):
    def __init__(self):
        print("BirdBad.__init__ called")
        AnimalBad.__init__(self)        # ❌ calling the parent directly, not super()
        self.lays_eggs = True

class MammalBad(AnimalBad):
    def __init__(self):
        print("MammalBad.__init__ called")
        AnimalBad.__init__(self)        # ❌ calling the parent directly, not super()
        self.has_fur = True

class PlatypusBad(BirdBad, MammalBad):
    def __init__(self):
        print("PlatypusBad.__init__ called")
        BirdBad.__init__(self)          # ❌ calling directly instead of super()
        MammalBad.__init__(self)        # ❌ calling directly instead of super()

print("Creating a PlatypusBad instance:")
p_bad = PlatypusBad()


Creating a PlatypusBad instance:
PlatypusBad.__init__ called
BirdBad.__init__ called
AnimalBad.__init__ called
MammalBad.__init__ called
AnimalBad.__init__ called


### What just happened?

Look closely at the printed order: **`AnimalBad.__init__` was called TWICE** —
once via the `BirdBad` branch, and again via the `MammalBad` branch.

This happens because calling `AnimalBad.__init__(self)` directly **bypasses the MRO
entirely**. Each branch has no idea the other branch already initialised the shared
parent, so it does it again.

For a simple example like this, calling it twice is harmless — but in real code this
causes genuine bugs:

- A counter incremented in `__init__` would be incremented twice
- A database connection or file handle opened in `__init__` would be opened twice
- Any expensive setup logic runs twice, wasting time and resources
- In more complex diamonds, this can even cause some classes to be **skipped entirely**
  instead of duplicated, depending on the exact shape of the hierarchy



### The fix: always use `super()`, never call a parent class by name

| Approach | What happens in a diamond hierarchy |
|---|---|
| `ParentClass.__init__(self)` | Bypasses the MRO — shared ancestors can be called multiple times |
| `super().__init__()` | Follows the MRO — each ancestor's `__init__` runs **exactly once** |

The rule of thumb: **in multiple inheritance, every class in the hierarchy should call
`super().__init__()` — never the parent class by name directly** — so Python's MRO can
correctly coordinate the whole chain for you.


**What does this code do - explain it**

In [7]:
class Person:
    def __init__(self, name, age, **kwargs):
        super().__init__(**kwargs)          # cooperative — forwards anything left over
        self.name = name
        self.age = age

    def greet(self):
        return f"Hi, I'm {self.name}, {self.age} years old."


class Staff(Person):
    def __init__(self, employee_id, salary, **kwargs):
        super().__init__(**kwargs)          # only consumes its own keywords
        self.employee_id = employee_id
        self.salary = salary

    def work(self):
        return f"{self.name} (staff #{self.employee_id}) is working."


class Student(Person):
    def __init__(self, student_id, programme, **kwargs):
        super().__init__(**kwargs)
        self.student_id = student_id
        self.programme = programme

    def study(self):
        return f"{self.name} (student #{self.student_id}) is studying {self.programme}."


# ── The diamond: someone who is BOTH Staff and Student at once ──────────────
class TeachingAssistant(Staff, Student):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)          # just forwards everything along the MRO

    def describe_role(self):
        return f"{self.name} is both teaching staff and a student."

In [10]:
john = TeachingAssistant(
    name="John",
    age=31,
    employee_id="S102",
    salary=51000,
    student_id="ST450",
    programme="MSc Artificial Intelligence",)

# test john here


## 5. Summary

| Concept | What it means | How to use it |
|---|---|---|
| Single inheritance | One class inherits from one parent | `class Dog(Animal):` |
| Multiple inheritance | One class inherits from several parents | `class FlyingFish(Swimmer, Flyer):` |
| MRO | The order Python searches for a method across the hierarchy | `ClassName.__mro__` or `ClassName.mro()` |
| Diamond problem | A shared ancestor reachable via two different parent paths | Solved automatically by the MRO |
| `super()` | Delegates to the *next* class in the MRO, not a hardcoded parent | `super().__init__()`, `super().method()` |

### The one rule to remember

> **Always call `super().__init__()` in every class of a multi-class hierarchy.**
> Never call a specific parent class's method directly (e.g. `ParentClass.method(self)`)
> unless you have a very specific reason to bypass the MRO on purpose.